In [ ]:
# %% [markdown]
# # Phase 3: Feature Engineering for Geometric Analysis
#
# **Goal**: Create features that expose meaningful structure for distance metrics.

# %%
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data_loader import load_raw_data
from features import create_temporal_features, create_pollution_features, get_feature_matrix

print(" Feature Engineering for Geometric Analysis")

# %%
# Load cleaned data
df_clean = pd.read_csv('../data/processed/air_quality_cleaned.csv',
                       index_col='datetime', parse_dates=True)
print(f" Cleaned data: {df_clean.shape}")

# %%
# Create temporal features
df_temp = create_temporal_features(df_clean)
temporal_cols = [col for col in df_temp.columns if col not in df_clean.columns]

print(" Temporal Features Created:")
for col in temporal_cols:
    print(f"  • {col}")

# %%
# Create pollution features
df_feat = create_pollution_features(df_temp)
pollution_cols = [col for col in df_feat.columns if col not in df_temp.columns]

print(" Pollution Features Created:")
for col in pollution_cols:
    print(f"  • {col}")

# %%
# Get complete feature matrix
X_features, feature_names = get_feature_matrix(df_clean)
print(f"\n🎯 Final Feature Matrix: {X_features.shape}")
print(f"Total features: {len(feature_names)}")
print(f"\nFirst 15 features: {feature_names[:15]}")

# %%
# Visualize feature relationships
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Cyclic time features
ax1 = axes[0, 0]
if 'hour_sin' in X_features.columns and 'hour_cos' in X_features.columns:
    ax1.scatter(X_features['hour_sin'][::100],
               X_features['hour_cos'][::100],
               c=X_features.index.hour[::100],
               cmap='hsv', alpha=0.6, s=20)
    ax1.set_xlabel('hour_sin')
    ax1.set_ylabel('hour_cos')
    ax1.set_title('Cyclic Time Encoding', fontweight='bold')
    ax1.grid(True, alpha=0.3)

# 2. Pollutant ratios
ax2 = axes[0, 1]
if 'CO_to_NOx' in X_features.columns:
    X_features['CO_to_NOx'].plot(ax=ax2, linewidth=0.5, alpha=0.7)
    ax2.set_ylabel('CO/NOx Ratio')
    ax2.set_title('Combustion Signature (CO/NOx)', fontweight='bold')
    ax2.grid(True, alpha=0.3)

# 3. Weekend vs weekday
ax3 = axes[0, 2]
if 'is_weekend' in X_features.columns and 'CO(GT)' in X_features.columns:
    weekend_data = X_features[X_features['is_weekend'] == 1]['CO(GT)']
    weekday_data = X_features[X_features['is_weekend'] == 0]['CO(GT)']

    ax3.hist(weekday_data.dropna(), bins=50, alpha=0.5, label='Weekday', density=True)
    ax3.hist(weekend_data.dropna(), bins=50, alpha=0.5, label='Weekend', density=True)
    ax3.set_xlabel('CO(GT) Concentration')
    ax3.set_ylabel('Density')
    ax3.set_title('Weekend vs Weekday Distribution')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

# 4. Rate of change
ax4 = axes[1, 0]
if 'CO(GT)_diff' in X_features.columns:
    X_features['CO(GT)_diff'].plot(ax=ax4, linewidth=0.5, alpha=0.7)
    ax4.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    ax4.set_ylabel('ΔCO(GT) (1-hour difference)')
    ax4.set_title('Pollutant Rate of Change', fontweight='bold')
    ax4.grid(True, alpha=0.3)

# 5. Time of day categories
ax5 = axes[1, 1]
if all(col in X_features.columns for col in ['is_night', 'is_morning', 'is_afternoon', 'is_evening']):
    time_categories = ['Night', 'Morning', 'Afternoon', 'Evening']
    time_cols = ['is_night', 'is_morning', 'is_afternoon', 'is_evening']

    category_means = []
    for col in time_cols:
        if 'CO(GT)' in X_features.columns:
            mean_co = X_features[X_features[col] == 1]['CO(GT)'].mean()
            category_means.append(mean_co)

    bars = ax5.bar(time_categories, category_means,
                  color=['navy', 'skyblue', 'orange', 'darkred'])
    ax5.set_ylabel('Mean CO(GT)')
    ax5.set_title('Pollution by Time of Day', fontweight='bold')
    ax5.grid(True, alpha=0.3, axis='y')

    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax5.text(bar.get_x() + bar.get_width()/2., height + 0.05,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)

# 6. Feature correlation (subset)
ax6 = axes[1, 2]
# Select a subset of features for correlation
corr_features = feature_names[:10] if len(feature_names) >= 10 else feature_names
corr_matrix = X_features[corr_features].corr()

im = ax6.imshow(corr_matrix, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
ax6.set_xticks(range(len(corr_features)))
ax6.set_yticks(range(len(corr_features)))
ax6.set_xticklabels(corr_features, rotation=90, fontsize=8)
ax6.set_yticklabels(corr_features, fontsize=8)
ax6.set_title('Feature Correlation (First 10)', fontweight='bold')
plt.colorbar(im, ax=ax6, label='Correlation')

plt.suptitle('Feature Engineering: Creating Structure for Geometric Analysis',
            fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/03_feature_engineering.png', dpi=150, bbox_inches='tight')
plt.show()

# %%
# Save feature matrix
X_features.to_csv('../data/processed/feature_matrix.csv')
print("\n Feature matrix saved to ../data/processed/feature_matrix.csv")

# %%
# Geometric interpretation
print("\n GEOMETRIC INTERPRETATION OF ENGINEERED FEATURES:")
print("=" * 60)
print("1. Cyclic encodings (sin/cos): Preserve circular geometry of time.")
print("   Euclidean distance between 23:59 and 00:01 should be small.")
print("\n2. Pollutant ratios: Create scale-invariant features.")
print("   Cosine distance on ratios finds similar combustion regimes.")
print("\n3. Time-of-day indicators: Create categorical structure.")
print("   May form natural clusters in high-dimensional space.")
print("\n4. Rate-of-change features: Capture dynamical system behavior.")
print("   Important for time-series-specific distance metrics (DTW).")
print("\nNext: Distance geometry analysis will reveal how these features")
print("behave under different similarity measures.")